In [0]:
# =============================================================================
# Notebook  : TEST_silver_layer
# Location  : /HGI-Lakehouse-Pipeline/05_Tests/TEST_silver_layer
# Purpose   : Validates all silver CDF tables + all Map layer output tables.
#             Covers ALL spec DQ gates from the client PDFs.
# Usage     : Set customer_id widget → Run All
# =============================================================================

In [0]:
# CELL 1
%run ../config/pipeline_config

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

from datetime import datetime, date
from pyspark.sql import functions as F

results = []

def test(name, fn):
    try:
        passed, detail = fn()
        results.append({"test": name, "passed": passed, "detail": detail})
        mark = "✅" if passed else "❌"
        print(f"  {mark} {'PASS' if passed else 'FAIL'} | {name}")
        if not passed:
            print(f"         Detail: {detail}")
    except Exception as e:
        results.append({"test": name, "passed": False, "detail": str(e)})
        print(f"  💥 ERROR | {name}: {e}")

def cnt(t):
    return spark.table(t).count()

print(f"=== Silver Layer Test Suite ===")
print(f"  customer_id : {customer_id}")
print(f"  Started     : {datetime.now()}")
print("=" * 65)

# Expected tables
SILVER_BASE_TABLES  = ["accounts", "contacts", "opportunities", "tasks",
                        "campaigns", "campaign_members", "users", "events"]
MAP_OUTPUT_TABLES   = ["contacts_all", "accounts_all", "crm_events", "mapped_events",
                        "contacts_to_accounts", "accounts_attributes", "contacts_attributes",
                        "email_events_mapped", "domain_events_mapped",
                        "mk_account_events_mapped", "email_audience", "email_model_conversion"]

# ── SECTION 1: Infrastructure ────────────────────────────────────────────────
print("\n─── Infrastructure ───")

def test_silver_schema_exists():
    schemas = [r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN {silver_catalog}").collect()]
    return silver_schema in schemas, f"'{sv}' {'found' if silver_schema in schemas else 'NOT FOUND'}"
test("Silver schema exists", test_silver_schema_exists)

def test_all_silver_base_tables_exist():
    missing = []
    for tbl in SILVER_BASE_TABLES:
        try:
            spark.table(f"{sv}.{tbl}")
        except Exception:
            missing.append(tbl)
    return len(missing) == 0, f"Missing: {missing}" if missing else "All 8 silver base tables exist"
test("All 8 silver base tables exist", test_all_silver_base_tables_exist)

def test_all_map_tables_exist():
    missing = []
    for tbl in MAP_OUTPUT_TABLES:
        try:
            spark.table(f"{sv}.{tbl}")
        except Exception:
            missing.append(tbl)
    return len(missing) == 0, f"Missing: {missing}" if missing else "All map tables exist"
test("All map output tables exist", test_all_map_tables_exist)

# ── SECTION 2: Silver CDF Base Tables ────────────────────────────────────────
print("\n─── Silver CDF Base Tables (02a) ───")

def test_all_silver_base_have_data():
    empty = []
    for tbl in SILVER_BASE_TABLES:
        try:
            n = cnt(f"{sv}.{tbl}")
            if n == 0:
                empty.append(tbl)
        except Exception as e:
            empty.append(f"{tbl}: {e}")
    return len(empty) == 0, f"Empty: {empty}" if empty else "All 8 silver base tables have data"
test("All 8 silver base tables have data", test_all_silver_base_have_data)

def test_no_cdf_columns_in_silver():
    cdf_meta = {"_change_type", "_commit_version", "_commit_timestamp"}
    for tbl in SILVER_BASE_TABLES:
        try:
            cols = {c.lower() for c in spark.table(f"{sv}.{tbl}").columns}
            found = cdf_meta & cols
            if found:
                return False, f"{tbl}: CDF columns persisted: {found}"
        except Exception:
            pass
    return True, "No CDF internal columns persisted in silver"
test("[KEY] CDF internal columns not persisted in silver", test_no_cdf_columns_in_silver)

def test_silver_contacts_email_lowercase():
    df = spark.table(f"{sv}.contacts")
    if "email" not in df.columns:
        return True, "email not in contacts yet"
    bad = df.filter(
        F.col("email").isNotNull() & (F.col("email") != F.lower(F.col("email")))
    ).count()
    return bad == 0, f"{bad} contacts with uppercase email"
test("[SPEC GATE] silver.contacts email is lowercase", test_silver_contacts_email_lowercase)

def test_silver_users_email_lowercase():
    df = spark.table(f"{sv}.users")
    if "email" not in df.columns:
        return True, "email not in users yet"
    bad = df.filter(
        F.col("email").isNotNull() & (F.col("email") != F.lower(F.col("email")))
    ).count()
    return bad == 0, f"{bad} users with uppercase email"
test("silver.users email is lowercase", test_silver_users_email_lowercase)

def test_no_null_silver_ids():
    issues = []
    for tbl in SILVER_BASE_TABLES:
        bad = spark.table(f"{sv}.{tbl}").filter(F.col("id").isNull()).count()
        if bad > 0:
            issues.append(f"{tbl}: {bad}")
    return len(issues) == 0, "; ".join(issues) if issues else "No null IDs in silver base tables"
test("No null IDs in silver base tables", test_no_null_silver_ids)

def test_custom_metadata_non_null():
    issues = []
    for tbl in SILVER_BASE_TABLES:
        df = spark.table(f"{sv}.{tbl}")
        if "custom_metadata" in df.columns:
            bad = df.filter(F.col("custom_metadata").isNull()).count()
            if bad > 0:
                issues.append(f"{tbl}: {bad} null custom_metadata")
    return len(issues) == 0, "; ".join(issues) if issues else "custom_metadata non-null"
test("custom_metadata MAP is non-null", test_custom_metadata_non_null)

# ── SECTION 3: contacts_all + accounts_all ───────────────────────────────────
print("\n─── Materialized Views: contacts_all, accounts_all ───")

def test_contacts_all_cols():
    req  = {"id", "email", "domain", "source_system", "tenant"}
    cols = set(spark.table(f"{sv}.contacts_all").columns)
    miss = req - cols
    return len(miss) == 0, f"Missing: {miss}" if miss else "All required columns present"
test("contacts_all has required columns", test_contacts_all_cols)

def test_accounts_all_cols():
    req  = {"id", "domain", "source_system", "tenant"}
    cols = set(spark.table(f"{sv}.accounts_all").columns)
    miss = req - cols
    return len(miss) == 0, f"Missing: {miss}" if miss else "All required columns present"
test("accounts_all has required columns", test_accounts_all_cols)

def test_contacts_all_no_null_id():
    bad = spark.table(f"{sv}.contacts_all").filter(F.col("id").isNull()).count()
    return bad == 0, f"{bad} null IDs"
test("contacts_all: no null IDs", test_contacts_all_no_null_id)

def test_accounts_domain_lowercase():
    bad = spark.table(f"{sv}.accounts_all").filter(
        F.col("domain").isNotNull() & (F.col("domain") != F.lower(F.col("domain")))
    ).count()
    return bad == 0, f"{bad} accounts_all with uppercase domain"
test("[SPEC GATE] accounts_all domain is lowercase", test_accounts_domain_lowercase)

# ── SECTION 4: crm_events + mapped_events ────────────────────────────────────
print("\n─── crm_events + mapped_events ───")

def test_crm_events_has_data():
    n = cnt(f"{sv}.crm_events")
    return n > 0, f"{n:,} rows in crm_events"
test("crm_events has data", test_crm_events_has_data)

def test_meta_event_100pct_nonnull():
    """[SPEC GATE] 100% of mapped_events must have non-null meta_event"""
    total = cnt(f"{sv}.mapped_events")
    null_meta = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE meta_event IS NULL"
    ).collect()[0][0]
    pct = round(100 * (total - null_meta) / max(total, 1), 1)
    threshold = DQ_THRESHOLDS["meta_event_nonnull_pct"]
    return null_meta == 0, (
        f"{null_meta} null meta_events out of {total:,} ({pct}% coverage). "
        f"Spec gate = {threshold}%"
    )
test("[SPEC GATE] meta_event 100% non-null", test_meta_event_100pct_nonnull)

def test_meta_event_lowercase_no_spaces():
    """meta_event must be lowercase, underscore-separated, no spaces or colons"""
    bad = spark.table(f"{sv}.mapped_events").filter(
        F.col("meta_event").rlike(r"[A-Z\s:]")
    ).count()
    return bad == 0, (
        f"{bad} events with uppercase / spaces / colons in meta_event. "
        "Should be: email_sent, call, page_view — NOT 'Email: Re:'"
    )
test("[SPEC GATE] meta_event is lowercase underscore-separated", test_meta_event_lowercase_no_spaces)

# ── SECTION 5: contacts_to_accounts ─────────────────────────────────────────
print("\n─── contacts_to_accounts (3-phase) ───")

def test_c2a_has_data():
    n = cnt(f"{sv}.contacts_to_accounts")
    return n > 0, f"{n:,} rows"
test("contacts_to_accounts has data", test_c2a_has_data)

def test_c2a_no_orphan_contact_ids():
    """[SPEC GATE] Every contact_id must exist in contacts"""
    orphans = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.contacts_to_accounts c2a
        LEFT JOIN {sv}.contacts_all c ON c2a.contact_id = c.id
        WHERE c.id IS NULL
    """).collect()[0][0]
    return orphans == 0, f"{orphans} orphan contact_ids. Spec gate = 0"
test("[SPEC GATE] contacts_to_accounts: no orphan contact_ids", test_c2a_no_orphan_contact_ids)

def test_c2a_no_orphan_crm_account_ids():
    """[SPEC GATE] crm_defined links must point to real accounts"""
    orphans = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.contacts_to_accounts c2a
        LEFT JOIN {sv}.accounts_all a ON c2a.account_id = a.id
        WHERE a.id IS NULL AND c2a.match_type = 'crm_defined'
    """).collect()[0][0]
    return orphans == 0, f"{orphans} crm_defined links with no matching account. Spec gate = 0"
test("[SPEC GATE] contacts_to_accounts: no orphan account_ids (crm_defined)", test_c2a_no_orphan_crm_account_ids)

def test_c2a_linkage_coverage():
    """[SPEC GATE] ≥80% of contacts linked to an account"""
    total_contacts = cnt(f"{sv}.contacts_all")
    linked         = cnt(f"{sv}.contacts_to_accounts")
    pct            = round(100 * linked / max(total_contacts, 1), 1)
    threshold      = DQ_THRESHOLDS["c2a_linkage_pct"]
    return pct >= threshold, (
        f"{linked:,}/{total_contacts:,} = {pct}% linked. "
        f"Spec gate = ≥{threshold}%"
    )
test("[SPEC GATE] contacts_to_accounts linkage ≥80%", test_c2a_linkage_coverage)

def test_c2a_match_type_valid():
    valid = {"crm_defined", "domain_match", "domain_created", "free_email_match"}
    bad   = spark.table(f"{sv}.contacts_to_accounts").filter(
        ~F.col("match_type").isin(*valid)
    ).count()
    return bad == 0, f"{bad} rows with invalid match_type (valid: {valid})"
test("contacts_to_accounts match_type values valid", test_c2a_match_type_valid)

def test_c2a_no_self_links():
    bad = spark.table(f"{sv}.contacts_to_accounts").filter(
        F.col("contact_id") == F.col("account_id")
    ).count()
    return bad == 0, f"{bad} self-referential links"
test("No self-referential contact-to-account links", test_c2a_no_self_links)

# ── SECTION 6: Aggregation Tables ────────────────────────────────────────────
print("\n─── Aggregation Tables ───")

def test_domain_events_domain_lowercase():
    bad = spark.table(f"{sv}.domain_events_mapped").filter(
        F.col("domain").isNotNull() & (F.col("domain") != F.lower(F.col("domain")))
    ).count()
    return bad == 0, f"{bad} rows with uppercase domain in domain_events_mapped"
test("[SPEC GATE] domain_events_mapped domain is lowercase", test_domain_events_domain_lowercase)

def test_occurrences_positive():
    bad = spark.table(f"{sv}.email_events_mapped").filter(
        F.col("occurrences") <= 0
    ).count()
    return bad == 0, f"{bad} rows with occurrences ≤ 0"
test("email_events_mapped occurrences > 0", test_occurrences_positive)

# ── SECTION 7: Attribute Tables ──────────────────────────────────────────────
print("\n─── Attribute Tables ───")

def test_accounts_attributes_coverage():
    """[SPEC GATE] Every account must have an attributes row"""
    uncovered = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.accounts_all a
        LEFT JOIN {sv}.accounts_attributes aa ON a.id = aa.account_id
        WHERE aa.account_id IS NULL
    """).collect()[0][0]
    return uncovered == 0, (
        f"{uncovered} accounts without attributes row. Spec gate = 0"
    )
test("[SPEC GATE] accounts_attributes covers ALL accounts", test_accounts_attributes_coverage)

def test_attributes_schema():
    for tbl in ["accounts_attributes", "contacts_attributes"]:
        cols = set(spark.table(f"{sv}.{tbl}").columns)
        req  = {"is_paying", "is_excluded", "mrr"}
        miss = req - cols
        if miss:
            return False, f"{tbl}: missing columns: {miss}"
    return True, "is_paying, is_excluded, mrr present on both tables"
test("Attribute tables have required schema", test_attributes_schema)

# ── SECTION 8: Enrichment Tables (Augment layer — test if present) ───────────
print("\n─── Augment Tables (post-enrichment) ───")

def test_persons_email_lowercase():
    try:
        bad = spark.sql(
            f"SELECT COUNT(*) FROM {sv}.persons WHERE mk_email != LOWER(mk_email)"
        ).collect()[0][0]
        return bad == 0, f"{bad} uppercase emails in persons"
    except Exception:
        return True, "persons table not yet created (enrichment not run)"
test("[SPEC GATE] persons.mk_email is lowercase", test_persons_email_lowercase)

def test_persons_mk_email_unique():
    try:
        dupes = spark.sql(f"""
            SELECT COUNT(*) FROM (
                SELECT mk_email, COUNT(*) AS n FROM {sv}.persons
                GROUP BY mk_email HAVING n > 1
            )
        """).collect()[0][0]
        return dupes == 0, f"{dupes} duplicate mk_email in persons"
    except Exception:
        return True, "persons table not yet created"
test("[SPEC GATE] persons.mk_email is unique", test_persons_mk_email_unique)

def test_companies_domain_lowercase():
    try:
        bad = spark.sql(
            f"SELECT COUNT(*) FROM {sv}.companies WHERE mk_domain != LOWER(mk_domain)"
        ).collect()[0][0]
        return bad == 0, f"{bad} uppercase domains in companies"
    except Exception:
        return True, "companies table not yet created"
test("[SPEC GATE] companies.mk_domain is lowercase", test_companies_domain_lowercase)

def test_companies_mk_domain_unique():
    try:
        dupes = spark.sql(f"""
            SELECT COUNT(*) FROM (
                SELECT mk_domain, COUNT(*) AS n FROM {sv}.companies
                GROUP BY mk_domain HAVING n > 1
            )
        """).collect()[0][0]
        return dupes == 0, f"{dupes} duplicate mk_domain in companies"
    except Exception:
        return True, "companies table not yet created"
test("[SPEC GATE] companies.mk_domain is unique", test_companies_mk_domain_unique)

def test_mk_status_set():
    try:
        bad = spark.sql(
            f"SELECT COUNT(*) FROM {sv}.persons WHERE mk_status IS NULL"
        ).collect()[0][0]
        return bad == 0, f"{bad} persons with null mk_status"
    except Exception:
        return True, "persons table not yet created"
test("[SPEC GATE] mk_status set on all enriched records", test_mk_status_set)

# ── SECTION 9: Spec DQ Gate Summary ─────────────────────────────────────────
print("\n─── Spec Quality Gate Summary ───")
try:
    null_meta     = spark.sql(
        f"SELECT COUNT(*) FROM {sv}.mapped_events WHERE meta_event IS NULL"
    ).collect()[0][0]
    events_total  = cnt(f"{sv}.mapped_events")
    meta_pct      = round(100 * (events_total - null_meta) / max(events_total, 1), 1)

    total_contacts = cnt(f"{sv}.contacts_all")
    linked         = cnt(f"{sv}.contacts_to_accounts")
    linkage_pct    = round(100 * linked / max(total_contacts, 1), 1)

    uncov_accounts = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.accounts_all a
        LEFT JOIN {sv}.accounts_attributes aa ON a.id = aa.account_id
        WHERE aa.account_id IS NULL
    """).collect()[0][0]

    c2a_orphans = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.contacts_to_accounts c2a
        LEFT JOIN {sv}.contacts_all c ON c2a.contact_id = c.id
        WHERE c.id IS NULL
    """).collect()[0][0]

    gates = [
        ("contacts_to_accounts linkage",         linkage_pct,  DQ_THRESHOLDS["c2a_linkage_pct"],              "% (target ≥80%)"),
        ("meta_event coverage",                   meta_pct,     DQ_THRESHOLDS["meta_event_nonnull_pct"],       "% (target 100%)"),
        ("c2a orphan contact_ids",                c2a_orphans,  0,                                             " (target = 0)"),
        ("accounts_attributes uncovered accounts",uncov_accounts,0,                                            " (target = 0)"),
    ]

    for gate_name, actual, threshold, suffix in gates:
        if suffix.startswith("%"):
            ok = actual >= threshold
        else:
            ok = actual <= threshold
        mark = "✅" if ok else "❌"
        print(f"  {mark} {gate_name}: {actual}{suffix}")

except Exception as e:
    print(f"  Could not compute all DQ gates: {e}")

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
total  = len(results)
passed = sum(1 for r in results if r["passed"])
failed = total - passed

print(f"Results: {passed}/{total} passed, {failed} failed")
print(f"Finished: {datetime.now()}")

if failed > 0:
    print("\nFailing tests:")
    for r in results:
        if not r["passed"]:
            print(f"  ❌ {r['test']}")
            print(f"     {r['detail']}")
    raise Exception(f"Silver test suite: {failed} test(s) failed.")
else:
    print("\n✅ All silver tests passed!")